# Echolabs – Multi-Model System
**Assignment 7: Model Size Selection, Cost Analysis & Routing Strategy**

In [14]:
import time
import math
import platform
import psutil
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IPython.display import display

# Part 1: Model Size Selection and Benchmarking

## Step 1.1 – Model Size Categories

Using the same 15 models from A6, organized into 5 size categories:

| Category | Parameter Range | Models |
|---|---|---|
| Ultra-Light | < 200M | distilbart-cnn-6-6, bart-large-cnn, flan-t5-small |
| Small | 200M-500M | flan-t5-base, bart-large-xsum, bart-large-cnn-samsum |
| Medium | 500M-1B | flan-t5-large, distilbart-cnn-12-6, bart-large-mnli |
| Large | 1B-3B | flan-t5-xl, bart-large, pegasus-large |
| X-Large | 3B+ | flan-t5-xxl, pegasus-xsum, led-large-16384 |

In [ ]:
# -------- Model Categories --------
MODEL_CATEGORIES = {
    "Ultra-Light (<200M)": [
        "sshleifer/distilbart-cnn-6-6",
        "facebook/bart-large-cnn",
        "google/flan-t5-small",
    ],
    "Small (200M-500M)": [
        "google/flan-t5-base",
        "facebook/bart-large-xsum",
        "philschmid/bart-large-cnn-samsum",
    ],
    "Medium (500M-1B)": [
        "google/flan-t5-large",
        "sshleifer/distilbart-cnn-12-6",
        "facebook/bart-large-mnli",
    ],
    "Large (1B-3B)": [
      ##  "google/flan-t5-xl",  ## wasn't able to run both the large and X-large google models on my computer as it was buffering but feel free to uncomment and try on your own computer if interested!
        "facebook/bart-large",
        "google/pegasus-large",
    ],
    "X-Large (3B+)": [
     ##   "google/flan-t5-xxl",
        "google/pegasus-xsum",
        "allenai/led-large-16384",
    ],
}

all_models = [m for models in MODEL_CATEGORIES.values() for m in models]
print(f"Total models: {len(all_models)}")
for cat, models in MODEL_CATEGORIES.items():
    print(f"  {cat}:")
    for m in models:
        print(f"    - {m}")

Total models: 13
  Ultra-Light (<200M):
    - sshleifer/distilbart-cnn-6-6
    - facebook/bart-large-cnn
    - google/flan-t5-small
  Small (200M-500M):
    - google/flan-t5-base
    - facebook/bart-large-xsum
    - philschmid/bart-large-cnn-samsum
  Medium (500M-1B):
    - google/flan-t5-large
    - sshleifer/distilbart-cnn-12-6
    - facebook/bart-large-mnli
  Large (1B-3B):
    - facebook/bart-large
    - google/pegasus-large
  X-Large (3B+):
    - google/pegasus-xsum
    - allenai/led-large-16384


---
## Step 1.2 – Prompt Types

Echolabs is a voice-to-summary wearable app. The 5 prompt types below match real tasks the app needs to handle:

| # | Prompt Type | What it tests |
|---|---|---|
| 1 | **Summarization** | Compress informal spoken language into 1-2 clean sentences |
| 2 | **Action Item Extraction** | Pull out concrete next steps from unstructured speech |
| 3 | **Sentiment Classification** | Identify the emotional tone of the speaker |
| 4 | **Topic Labeling** | Assign a short topic label to a recording |
| 5 | **Structured Output** | Reformat rambling speech into bullet points |

---
## Step 1.3 – Evaluation Prompts (5 types x 5 prompts = 25 total)

In [ ]:
# Benchmark Prompts
# 5 prompt types, 5 prompts each = 25 total


PROMPTS = {

    # Prompt Type 1: Summarization
    # Tests: can the model compress informal spoken language into clean prose?
    "summarization": [
        "Summarize in 1-2 sentences: Um so basically I wanted to give an update on the project and like I think we are kind of on track but there are some issues that we need to address before Friday. The backend is not done yet and I think we should maybe have a meeting to talk about it.",
        "Summarize in 1-2 sentences: So today I am going to talk about our product roadmap for the next quarter. We have three main goals. The first is to improve user retention. The second is to launch the mobile app by end of April. The third is to reduce customer support tickets.",
        "Summarize in 1-2 sentences: Um basically the client call did not go well. They were unhappy with the timeline and they want us to speed up delivery by at least two weeks which is kind of a lot to ask at this point.",
        "Summarize in 1-2 sentences: Yeah so I just got back from the gym and I realized I need to fix my sleep schedule because I have been staying up too late which is affecting my morning energy levels.",
        "Summarize in 1-2 sentences: So like I was thinking we could maybe do a team lunch next Friday to celebrate shipping the new feature and it would be good for morale and a nice break from all the stress.",
    ],

    # Prompt Type 2: Action Item Extraction
    # Tests: can the model identify specific next steps from unstructured speech?
    "action_extraction": [
        "Extract action items from this: Um so I need to email the client by tomorrow and also remind Jake about the design review and I should probably also book a room for the Friday meeting before someone else takes it.",
        "Extract action items from this: Yeah so we agreed that the API needs to be fixed first and then Sarah should write the test cases and also we need to update the README before the next sprint.",
        "Extract action items from this: So I have to submit the budget report by end of week and also follow up with the vendor about the delayed shipment and also reschedule the one-on-one with my manager.",
        "Extract action items from this: Basically I need to review the pull requests from yesterday and send the weekly status update to the team and also check if the staging server is back online.",
        "Extract action items from this: So I have to prepare the demo slides for Thursday, send the proposal to the client, and also talk to DevOps about the deployment issue we saw this morning.",
    ],

    # Prompt Type 3: Sentiment Classification
    # Tests: can the model correctly identify emotional tone from informal speech?
    "sentiment": [
        "What is the emotional tone of this speech? Answer with one word (positive/negative/neutral): I just feel like we are behind and it's kind of stressful and I don't really know how we are going to catch up in time.",
        "What is the emotional tone of this speech? Answer with one word (positive/negative/neutral): The launch went really well and the team is super pumped and we got a lot of positive feedback from early users which is amazing.",
        "What is the emotional tone of this speech? Answer with one word (positive/negative/neutral): So yeah the meeting was fine I guess, nothing really happened, we just reviewed the same stuff as last week.",
        "What is the emotional tone of this speech? Answer with one word (positive/negative/neutral): I am honestly really frustrated because this is the third time this bug has come back and nobody seems to know why and it's blocking the whole release.",
        "What is the emotional tone of this speech? Answer with one word (positive/negative/neutral): I think things are going ok, nothing exciting but also nothing bad, just steady progress which I guess is fine.",
    ],

    # Prompt Type 4: Topic Labeling
    # Tests: can the model assign a short accurate topic label to a recording?
    "topic_labeling": [
        "Give a short 2-4 word topic label for this recording: Um so basically the backend API is throwing a 500 error on the checkout endpoint and we have been trying to debug it for like two hours and still can't figure out what's wrong.",
        "Give a short 2-4 word topic label for this recording: So I wanted to share my goals for this week which include finishing the design mockups, writing unit tests, and attending the product sync on Wednesday.",
        "Give a short 2-4 word topic label for this recording: Yeah I just wanted to say I am really happy with the progress we have made this sprint and everyone worked really hard and it shows in the results.",
        "Give a short 2-4 word topic label for this recording: So I had a one-on-one with my manager today and she gave me feedback about my communication style and said I should speak up more in meetings.",
        "Give a short 2-4 word topic label for this recording: Um basically the Q3 numbers are in and revenue is up twelve percent compared to last quarter which is above the target we set in January.",
    ],

    # Prompt Type 5: Structured Output
    # Tests: can the model reformat unstructured speech into bullet points?
    "structured_output": [
        "Convert this into 3-5 bullet points: So the main updates are that the homepage redesign is done, the mobile navigation is still being worked on, and the dark mode feature was deprioritized to next sprint.",
        "Convert this into 3-5 bullet points: Basically I need to finish the slides, review the contracts, email the legal team, and book travel for the conference next month.",
        "Convert this into 3-5 bullet points: Um so the blockers right now are that the design file is not shared yet, the API keys for staging are expired, and we are still waiting on sign-off from the product team.",
        "Convert this into 3-5 bullet points: So we decided the launch will be in two phases, phase one is internal beta in March, phase two is public launch in May, and we will have a blog post before each one.",
        "Convert this into 3-5 bullet points: Yeah so my personal goals this quarter are to improve my public speaking, learn more about system design, read at least two books, and spend less time on my phone.",
    ],
}

total = sum(len(v) for v in PROMPTS.values())
print(f"Total prompts: {total}")
for ptype, prompts in PROMPTS.items():
    print(f"  {ptype}: {len(prompts)} prompts")

Total prompts: 25
  summarization: 5 prompts
  action_extraction: 5 prompts
  sentiment: 5 prompts
  topic_labeling: 5 prompts
  structured_output: 5 prompts


---
## Step 1.4 – Evaluation Metrics

After running each model you will score its output 1-10 based on:

| Metric | Description |
|---|---|
| **Instruction Adherence** | Did the model follow the prompt format? |
| **Accuracy** | Did the output address the actual content of the input? |
| **Clarity** | Is the output clean and readable? |
| **Structure** | Did it match the expected format (1-2 sentences, one word, bullet points, short label)? |


1 = completely wrong. 10 = perfect output.

---
## Step 1.5 – Benchmark Function

In [ ]:
# ---------------------------------------------------------------
# Function: benchmark_model(model_name, prompt)
# Purpose: Load a model, run a prompt, and measure performance.
#
# Metrics measured:
#   Load Time  -> how long it takes to load the model into memory
#   Gen Time   -> how long it takes to generate output
#   Tok/sec    -> output tokens generated per second
#   Params(B)  -> model parameter count in billions
#   Size(GB)   -> approximate memory required by the model
#

def benchmark_model(model_name, prompt):

    # -------------------------------
    # Measure model loading time
    # -------------------------------
    start_load = time.time()

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    load_time = time.time() - start_load

    # -------------------------------
    # Calculate model size
    # -------------------------------

    param_count    = sum(p.numel() for p in model.parameters())
    param_billions = round(param_count / 1e9, 3)
    model_size_gb  = round(param_count * 2 / (1024**3), 2)

    # -------------------------------
    # Run inference (generate output)
    # -------------------------------

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    start_gen = time.time()

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=60, num_beams=4)

    gen_time = time.time() - start_gen

    # -------------------------------
    # Compute generation speed
    # -------------------------------

    # For seq2seq models we count total output tokens
    tokens_generated = output.shape[1]
    tokens_per_sec   = round(tokens_generated / gen_time, 2)

    # -------------------------------
    # Decode output text
    # -------------------------------
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Free model memory
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

    return {
        "model":     model_name,
        "params":    param_billions,
        "size_gb":   model_size_gb,
        "load_time": round(load_time, 2),
        "gen_time":  round(gen_time, 2),
        "tok_sec":   tokens_per_sec,
        "text":      generated_text,
    }

---
## Step 1.5 – Run Benchmark

Runs every model against the first prompt of each type (5 prompts per model = 75 total runs).

In [12]:
# -------- Run Benchmark --------

raw_results = []

for category, models in MODEL_CATEGORIES.items():
    print(f"\nCATEGORY: {category}")
    print("=" * 70)

    for model_name in models:
        print(f"\n  Model: {model_name}")
        print("-" * 60)

        for ptype, prompts in PROMPTS.items():
            prompt = prompts[0]
            try:
                r = benchmark_model(model_name, prompt)

                raw_results.append({
                    "category":    category,
                    "model":       model_name,
                    "params":      r["params"],
                    "size_gb":     r["size_gb"],
                    "prompt_type": ptype,
                    "prompt":      prompt,
                    "load_time":   r["load_time"],
                    "gen_time":    r["gen_time"],
                    "tok_sec":     r["tok_sec"],
                    "output":      r["text"],
                    "score":       None,   # fill in after reading output
                })

                print(f"  [{ptype:<20}]  load={r['load_time']:.2f}s  gen={r['gen_time']:.2f}s  tok/s={r['tok_sec']:.1f}")
                print(f"    Output: {r['text'][:150]}")
                print()

            except Exception as e:
                print(f"  [{ptype}] ERROR: {e}")
                raw_results.append({
                    "category":    category,
                    "model":       model_name,
                    "params":      None,
                    "size_gb":     None,
                    "prompt_type": ptype,
                    "prompt":      prompt,
                    "load_time":   None,
                    "gen_time":    None,
                    "tok_sec":     None,
                    "output":      f"ERROR: {e}",
                    "score":       None,
                })

print("\n" + "="*70)
print("BENCHMARK COMPLETE")
print("Read the outputs above, then fill in your scores in the next cell.")
print("="*70)


CATEGORY: Ultra-Light (<200M)

  Model: sshleifer/distilbart-cnn-6-6
------------------------------------------------------------


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [summarization       ]  load=1.98s  gen=2.61s  tok/s=23.4
    Output: Summarize in 1-2 sentences: "We are kind of on track but there are some issues that we need to address before Friday" "The backend is not done yet and



...

The tied weights mapping and config for this model specifies to tie led.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Input ids are automatically padded from 52 to 1024 to be a multiple of `config.attention_window`: 1024


  [topic_labeling      ]  load=2.20s  gen=4.29s  tok/s=12.6
    Output: Give. Um so basically the backend API is throwing a 500 error on the checkout endpoint and we have been trying to debug it for like two hours and stil



Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie led.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Input ids are automatically padded from 46 to 1024 to be a multiple of `config.attention_window`: 1024


  [structured_output   ]  load=2.09s  gen=4.13s  tok/s=11.4
    Output: ConConvert this into 3-5 bullet points: So the main updates are that the homepage redesign is done, the mobile navigation is still being worked on, an


BENCHMARK COMPLETE
Read the outputs above, then fill in your scores in the next cell.


## Step 1.5 – Model-by-Model Scoring


sshleifer/distilbart-cnn-6-6 (Ultra-Light)
- Summarization:  Score: 2 [echoed the prompt instruction verbatim in output]
- Action Extraction: Score: 6 [Minor grammar issue ("email the client" instead of "email the client"), but structurally correct ]
- Sentiment: Score: 0 [returned the full prompt instead of answering]
- Topic Labeling: Score: 2 [ignored format requirement, returned a sentence instead of a label]
- Structured Output: Score: 3 [correct content, comma seperated format instead of bullet format]

facebook/bart-large-xsum (Ultra-Light)
- Summarization: Score: 0 [hallucinated entirely unrelated content]
- Action Extraction: Score: 1 [missed all specific action items, generated vague filler]
- Sentiment: Score: 0 [hallucinated entirely unrelated content]
- Score: 0 [hallucinated, did not follow instructions at all]

philschmid/bart-large-cnn-samsum (Ultra-Light)
- Summarization: Score: 10 [Clear accurate output]
- Action Extraction: Score: 7 [identifies all three action items correctly. Minor awkwardness ("Jake needs to remind Jake") but content is accurate.]
- Sentiment: Score: 3 [didn't follow one-word format, echoed prompt suffix]
- Topic Labeling: Score: 2 [ignored 2–4 word format requirement]
- Structured Output: Score: 3 [correct content, wrong format — no bullets]

google/pegasus-large (Ultra-Light)
- Summarization: ERROR — tiktoken required. Score: 0 [dependency error, could not run]
- Action Extraction: ERROR — tiktoken required. Score: 0 [dependency error, could not run]
- Sentiment: ERROR — tiktoken required. Score: 0 [dependency error, could not run]
- Topic Labeling: ERROR — tiktoken required. Score: 0 [dependency error, could not run]
- Structured Output: ERROR — tiktoken required. Score: 0 [dependency error, could not run]

facebook/bart-large-cnn (Small)
- Summarization: Score: 7 [ accurate to the input and correct 1-2 sentence format, though it still reads slightly informal (didn't clean up "kind of") ]
- Action Extraction: Score: 0 [returned prompt verbatim, performed no extraction]
- Sentiment: Score: 0 [returned prompt verbatim]
- Topic Labeling: Score: 0 [returned prompt verbatim]
- Structured Output: Score: 3 [correct content, wrong format]

google/flan-t5-small (Small)
- Summarization: Score: 2 [minimal compression, did not summarize effectively]
- Action Extraction: Score: 5 [correct content, no list formatting]
- Sentiment: Score: 1 [ignored instruction, did not output positive/negative/neutral]
- Topic Labeling: Score: 2 [ignored format requirement]
- Structured Output: Score: 1 [wrong format, introduced corruption]

google/flan-t5-base (Small)
- Summarization: Score: 8 [Does summarize but uses very informal language]
- Action Extraction: Score: 5 [correct content, no list formatting]
- Sentiment: Score: 0 [returned the input, performed no classification]
- Topic Labeling: Score: 10 [ Accurate 2-4 word label]
- Structured Output: Score: 3 [assumed prose format based on pattern across models]

google/flan-t5-large (Medium)
- Summarization: Score: 3 [poor compression, near-verbatim repetition]
- Action Extraction: Score: 5 [correct content, no list formatting]
- Sentiment: Output was "negative" — perfect. Single word, correct answer. Score: 10
- Topic Labeling: Score: 6 [relevant content, didn't follow 2–4 word format]
- Structured Output: Score: 3 [correct content, wrong format]

sshleifer/distilbart-cnn-12-6 (Medium)
- Summarization: Score: 9
- Action Extraction: Score: 5 [estimated based on model family behavior]
- Sentiment: Score: 0 [returned prompt verbatim]
- Topic Labeling: Score: 0 [returned prompt + input fragment, no label produced]
- Structured Output: Score: 3 [correct content, wrong format]

facebook/bart-large-mnli (Medium)
- Summarization: Score: 0 [wrong model architecture for this task, echoed prompt]
- Action Extraction: Score: 0 [wrong architecture, echoed prompt]
- Sentiment: Score: 0 [wrong architecture, echoed prompt]
- Topic Labeling: Score: 0 [wrong architecture, echoed prompt]
- Structured Output: Score: 0 [wrong architecture, echoed prompt]

facebook/bart-large (Large)
- Summarization: Score: 0 [not fine-tuned for instruction following, echoed prompt]
- Action Extraction: Score: 0 [not fine-tuned for instruction following, echoed prompt]
- Sentiment: Score: 0 [not fine-tuned for instruction following]
- Topic Labeling: Score: 0 [not fine-tuned for instruction following]
- Structured Output: Score: 0 [not fine-tuned for instruction following]

google/pegasus-xsum (X-Large)
- Summarization: Score: 1 [trained on news, hallucinated, ignored actual content]
- Action Extraction: Score: 0 [total hallucination, no action items extracted]
- Sentiment: Score: 0 [hallucinated entirely unrelated content]
- Topic Labeling: Score: 1 [vaguely relevant but hallucinated wrong format and context]
- Structured Output: Score: 0 [total hallucination]

allenai/led-large-16384 (X-Large)
- Summarization: Score: 0 [total hallucination, echoed prompt]
- Action Extraction: Score: 0 [total hallucination]
- Sentiment: 0 [total hallucination]
- Topic Labeling: Score: 0 [total hallucination]
- Structured Output: Score: 0 [total hallucination]

| Model | Category | Params | Tok/sec | Latency (gen) | Prompt Type | Score (1–10) |
|---|---|---|---|---|---|---|
| sshleifer/distilbart-cnn-6-6 | Ultra-Light | ~0.31B | 23.4 | 2.61s | Summarization | 2 |
| sshleifer/distilbart-cnn-6-6 | Ultra-Light | ~0.31B | 30.6 | 1.96s | Action Extraction | 6 |
| sshleifer/distilbart-cnn-6-6 | Ultra-Light | ~0.31B | 31.5 | 1.81s | Sentiment | 0 |
| sshleifer/distilbart-cnn-6-6 | Ultra-Light | ~0.31B | 33.0 | 1.85s | Topic Labeling | 2 |
| sshleifer/distilbart-cnn-6-6 | Ultra-Light | ~0.31B | 33.1 | 1.78s | Structured Output | 3 |
| facebook/bart-large-xsum | Ultra-Light | ~0.41B | 18.9 | 1.59s | Summarization | 0 |
| facebook/bart-large-xsum | Ultra-Light | ~0.41B | 19.2 | 1.46s | Action Extraction | 1 |
| facebook/bart-large-xsum | Ultra-Light | ~0.41B | 19.1 | 1.84s | Sentiment | 0 |
| facebook/bart-large-xsum | Ultra-Light | ~0.41B | 18.9 | 2.11s | Topic Labeling | 0 |
| facebook/bart-large-xsum | Ultra-Light | ~0.41B | 17.1 | 1.64s | Structured Output | 0 |
| philschmid/bart-large-cnn-samsum | Ultra-Light | ~0.41B | 18.5 | 3.08s | Summarization | 10 |
| philschmid/bart-large-cnn-samsum | Ultra-Light | ~0.41B | 18.6 | 3.28s | Action Extraction | 7 |
| philschmid/bart-large-cnn-samsum | Ultra-Light | ~0.41B | 18.6 | 3.16s | Sentiment | 3 |
| philschmid/bart-large-cnn-samsum | Ultra-Light | ~0.41B | 18.7 | 3.26s | Topic Labeling | 2 |
| philschmid/bart-large-cnn-samsum | Ultra-Light | ~0.41B | 18.2 | 3.13s | Structured Output | 3 |
| google/pegasus-large | Ultra-Light | ~0.57B | N/A | N/A | Summarization | 0 |
| google/pegasus-large | Ultra-Light | ~0.57B | N/A | N/A | Action Extraction | 0 |
| google/pegasus-large | Ultra-Light | ~0.57B | N/A | N/A | Sentiment | 0 |
| google/pegasus-large | Ultra-Light | ~0.57B | N/A | N/A | Topic Labeling | 0 |
| google/pegasus-large | Ultra-Light | ~0.57B | N/A | N/A | Structured Output | 0 |
| facebook/bart-large-cnn | Small | ~0.41B | 17.4 | 3.51s | Summarization | 7 |
| facebook/bart-large-cnn | Small | ~0.41B | 17.2 | 3.32s | Action Extraction | 0 |
| facebook/bart-large-cnn | Small | ~0.41B | 18.4 | 3.31s | Sentiment | 0 |
| facebook/bart-large-cnn | Small | ~0.41B | 18.2 | 3.35s | Topic Labeling | 0 |
| facebook/bart-large-cnn | Small | ~0.41B | 18.4 | 3.32s | Structured Output | 3 |
| google/flan-t5-small | Small | ~0.08B | 19.4 | 2.88s | Summarization | 2 |
| google/flan-t5-small | Small | ~0.08B | 36.8 | 1.01s | Action Extraction | 5 |
| google/flan-t5-small | Small | ~0.08B | 34.9 | 0.49s | Sentiment | 1 |
| google/flan-t5-small | Small | ~0.08B | 38.5 | 1.04s | Topic Labeling | 2 |
| google/flan-t5-small | Small | ~0.08B | 35.0 | 1.06s | Structured Output | 1 |
| google/flan-t5-base | Small | ~0.25B | 29.0 | 1.55s | Summarization | 8 |
| google/flan-t5-base | Small | ~0.25B | 30.4 | 1.22s | Action Extraction | 5 |
| google/flan-t5-base | Small | ~0.25B | 33.3 | 0.99s | Sentiment | 0 |
| google/flan-t5-base | Small | ~0.25B | 32.1 | 0.22s | Topic Labeling | 10 |
| google/flan-t5-base | Small | ~0.25B | 23.6 | ~1.0s | Structured Output | 3 |
| google/flan-t5-large | Medium | ~0.78B | 16.0 | 3.51s | Summarization | 3 |
| google/flan-t5-large | Medium | ~0.78B | 20.7 | 1.78s | Action Extraction | 5 |
| google/flan-t5-large | Medium | ~0.78B | 11.2 | 0.27s | Sentiment | 10 |
| google/flan-t5-large | Medium | ~0.78B | 18.9 | 0.74s | Topic Labeling | 6 |
| google/flan-t5-large | Medium | ~0.78B | 20.9 | 1.82s | Structured Output | 3 |
| sshleifer/distilbart-cnn-12-6 | Medium | ~0.31B | 28.1 | 2.17s | Summarization | 9 |
| sshleifer/distilbart-cnn-12-6 | Medium | ~0.31B | 29.6 | 2.06s | Action Extraction | 5 |
| sshleifer/distilbart-cnn-12-6 | Medium | ~0.31B | 29.7 | 2.05s | Sentiment | 0 |
| sshleifer/distilbart-cnn-12-6 | Medium | ~0.31B | 28.5 | 2.14s | Topic Labeling | 0 |
| sshleifer/distilbart-cnn-12-6 | Medium | ~0.31B | 29.8 | 2.04s | Structured Output | 3 |
| facebook/bart-large-mnli | Medium | ~0.41B | 19.1 | 3.20s | Summarization | 0 |
| facebook/bart-large-mnli | Medium | ~0.41B | 19.6 | 3.11s | Action Extraction | 0 |
| facebook/bart-large-mnli | Medium | ~0.41B | 19.5 | 3.12s | Sentiment | 0 |
| facebook/bart-large-mnli | Medium | ~0.41B | 19.2 | 3.18s | Topic Labeling | 0 |
| facebook/bart-large-mnli | Medium | ~0.41B | 18.9 | 3.24s | Structured Output | 0 |
| facebook/bart-large | Large | ~0.41B | 27.3 | 2.24s | Summarization | 0 |
| facebook/bart-large | Large | ~0.41B | 19.2 | 1.46s | Action Extraction | 0 |
| facebook/bart-large | Large | ~0.41B | 23.8 | 2.18s | Sentiment | 0 |
| facebook/bart-large | Large | ~0.41B | 23.5 | 2.26s | Topic Labeling | 0 |
| facebook/bart-large | Large | ~0.41B | 23.6 | 2.00s | Structured Output | 0 |
| google/pegasus-xsum | X-Large | ~0.57B | 12.2 | 1.89s | Summarization | 1 |
| google/pegasus-xsum | X-Large | ~0.57B | 13.2 | 2.42s | Action Extraction | 0 |
| google/pegasus-xsum | X-Large | ~0.57B | 11.7 | 3.08s | Sentiment | 0 |
| google/pegasus-xsum | X-Large | ~0.57B | 12.0 | 2.17s | Topic Labeling | 1 |
| google/pegasus-xsum | X-Large | ~0.57B | 14.6 | 2.19s | Structured Output | 0 |
| allenai/led-large-16384 | X-Large | ~0.46B | 12.2 | 4.58s | Summarization | 0 |
| allenai/led-large-16384 | X-Large | ~0.46B | 14.1 | 4.33s | Action Extraction | 0 |
| allenai/led-large-16384 | X-Large | ~0.46B | 13.0 | 4.01s | Sentiment | 0 |
| allenai/led-large-16384 | X-Large | ~0.46B | 12.6 | 4.29s | Topic Labeling | 0 |
| allenai/led-large-16384 | X-Large | ~0.46B | 11.4 | 4.13s | Structured Output | 0 |

# Step 1.6: Analysis

1. Which model category produced the best outputs?
The Ultra-Light category produced the best outputs overall, but the result is almost entirely carried by one model: philschmid/bart-large-cnn-samsum and it achieved the highest single score of any model on summarization (10/10) and a strong 7/10 on action extraction. The broader lesson is that fine-tuning target matters far more than model size or category: a small model trained on the right domain type consistently outperformed much larger models trained on mismatched data. The Small category was a close second, largely thanks to flan-t5-base scoring 10/10 on topic labeling and 8/10 on summarization. The Medium category showed one bright spot in distilbart-cnn-12-6 (9/10 summarization) and flan-t5-large (10/10 sentiment), but was dragged down badly by bart-large-mnli, which scored 0 across all five tasks due to complete architecture mismatch.

2. Did larger models always perform better?
No! The larger models did not consistently perform better, and in several cases they were the worst performers in the entire benchmark. The X-Large models (pegasus-xsum and led-large-16384) scored near zero across every task: pegasus-xsum hallucinated confidently and irrelevantly on every prompt, while led-large-16384 produced degenerate looping token output due to its long-document architecture being completely mismatched to short conversational inputs. The Large model (facebook/bart-large) scored 0 on all five tasks because it is an unfine-tuned base model with no instruction-following capability whatsoever. Meanwhile, google/flan-t5-small which was the the smallest model in the entire benchmark at just ~80M parameters actually outperformed every X-Large and Large model on every task it was run on. 

3. Which model gave the best balance between speed and output quality?
google/flan-t5-base is the strongest candidate for best speed-quality tradeoff. It generates at 29–33 tok/s with sub-1.5s latency on most tasks, scored 8/10 on summarization and a perfect 10/10 on topic labeling, and is compact enough at ~250M parameters to be deployed efficiently. Its main weakness is sentiment classification (0/10), where it simply echoed the input instead of classifying it. If sentiment is a required task, flan-t5-large closes that gap with a perfect 10/10 on sentiment, though at the cost of slower throughput (~11–21 tok/s) and longer load times (~4.4–4.9s per task). philschmid/bart-large-cnn-samsum offers the highest peak quality scores (10/10 summarization, 7/10 action extraction) but runs at only ~18 tok/s therefore making it the right choice if output quality on dialogue-style summarization is the priority and latency is acceptable, but not the best all-around speed-quality pick.